In [0]:
# ============================================================
# CREDIT CARD DEFAULT PREDICTION — XGBOOST BASELINE MODEL
# ============================================================

%pip install xgboost
# ------------------------------------------------------------
# 1. LOAD AND PREPARE DATA
# ------------------------------------------------------------

df = spark.table("CCC_FEATURES")
pdf = df.toPandas()

print("Dataset shape:", pdf.shape)

# Create indicator for customers without a positive average balance
pdf["NO_POSITIVE_BALANCE"] = (
    pdf["AVG_BILL_AMT"] <= 0
).astype(int)

# Replace undefined payment-to-balance ratios with 0
pdf["PAYMENT_TO_BALANCE"] = (
    pdf["PAYMENT_TO_BALANCE"].fillna(0)
)


# ------------------------------------------------------------
# 2. DEFINE TARGET AND PREDICTORS
# ------------------------------------------------------------

features = [
    "LIMIT_BAL",
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    "AGE",

    "PAY_1",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6",

    "BILL_AMT1",
    "BILL_AMT2",
    "BILL_AMT3",
    "BILL_AMT4",
    "BILL_AMT5",
    "BILL_AMT6",

    "PAY_AMT1",
    "PAY_AMT2",
    "PAY_AMT3",
    "PAY_AMT4",
    "PAY_AMT5",
    "PAY_AMT6",

    "AVG_BILL_AMT",
    "AVG_PAY_AMT",
    "AVG_UTILIZATION",
    "PAYMENT_TO_BALANCE",
    "NO_POSITIVE_BALANCE",
    "DELINQUENT_MONTHS",
    "MAX_DELINQUENCY",
    "BALANCE_CHANGE"
]

X = pdf[features]
y = pdf["dpnm"]

print("Predictors:", X.shape)
print("Target:", y.shape)


# ------------------------------------------------------------
# 3. CREATE TRAIN/TEST SPLIT
# ------------------------------------------------------------

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining observations:", len(X_train))
print("Testing observations:", len(X_test))
print("Training default rate:", round(y_train.mean(), 4))
print("Testing default rate:", round(y_test.mean(), 4))


# ------------------------------------------------------------
# 4. IMPORT AND TRAIN XGBOOST
# ------------------------------------------------------------

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

print("\nXGBoost model successfully trained.")


# ------------------------------------------------------------
# 5. GENERATE PREDICTIONS
# ------------------------------------------------------------

# Binary prediction (0 = no default, 1 = default)
xgb_pred = xgb_model.predict(X_test)

# Predicted probability of default
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]


# ------------------------------------------------------------
# 6. EVALUATE MODEL
# ------------------------------------------------------------

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

xgb_accuracy = accuracy_score(y_test, xgb_pred)
xgb_precision = precision_score(y_test, xgb_pred)
xgb_recall = recall_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred)
xgb_auc = roc_auc_score(y_test, xgb_prob)

print("\nXGBoost Results")
print("---------------------------")
print(f"Accuracy:  {xgb_accuracy:.4f}")
print(f"Precision: {xgb_precision:.4f}")
print(f"Recall:    {xgb_recall:.4f}")
print(f"F1 Score:  {xgb_f1:.4f}")
print(f"ROC-AUC:   {xgb_auc:.4f}")


# ------------------------------------------------------------
# CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(y_test, xgb_pred)

print("\nConfusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nTrue Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)


# ------------------------------------------------------------
# COMPARE WITH LOGISTIC REGRESSION BASELINE
# ------------------------------------------------------------

print("\nModel Comparison")
print("------------------------------------------------------")
print("Metric              Logistic Regression    XGBoost")
print("------------------------------------------------------")
print(f"Accuracy                 0.8035             {xgb_accuracy:.4f}")
print(f"Precision                0.6171             {xgb_precision:.4f}")
print(f"Recall                   0.2939             {xgb_recall:.4f}")
print(f"F1 Score                 0.3982             {xgb_f1:.4f}")
print(f"ROC-AUC                  0.7433             {xgb_auc:.4f}")
print("------------------------------------------------------")


### Initial XGBoost Results

#The baseline XGBoost model outperformed logistic regression across all
#evaluation metrics. ROC-AUC increased from 0.7433 to 0.7599, while recall
#increased from 29.39% to 37.38%.

#The improvement in recall is particularly important for the credit-risk
#application because the model identifies a greater proportion of customers
#who ultimately default. Precision also increased slightly from 61.71% to
#62.16%, indicating that the improvement in default detection did not require
#a deterioration in the accuracy of positive predictions.

#These results suggest that nonlinear relationships and interactions captured
#by gradient boosting provide additional predictive value beyond the linear
#baseline.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 75.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 190.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Dataset shape: (30000, 32)
Predictors: (30000, 31)
Target: (30000,)

Training observations: 24000
Testing observations: 6000
Training default rate: 0.2212
Testing default rate: 0.2212

XGBoost model successfully trained.

XGBoost Results
---------------------------
Accuracy:  0.8112
Precision: 0.6216
Recall:    0.3738
F1 Score:  0.4668
ROC-AUC:   0.7599

Confusion Matrix:
[[4371  302]
 [ 831  496]]

True Negatives: 4371
False Positives: 302
False Negatives: 831
True Positives: 496

Model Comparison
------------------------------------------------------
Metric              Logistic Regression    XGBoost
------------------------------------------------------